# Test Runner Notebook

Tests can be triggered from here manually before pushing (in future this may be possible to make part of a push hook so its automatic but that is out of scope for now).

GitActions triggers the unit tests within the github environment current for each interaction.

GitActions triggers this notebook via a pipeline within the databricks environment. This is only necessary for integration tests which need to happen within the databricks environment. (With DI and mocking we could do a certain amount of the integration tests in github environment but it is not necessary to run them there yet, though should still making the tests in this way).

## Test Env Setup

In [0]:
# MAGIC %pip install pytest
# because doesnt get pytest from toml???
import pytest
import sys
import os

# We need the catalog for our temp storage volume
# Get catalog from job
dbutils.widgets.text("job.catalog", "dev_catalog")
catalog = dbutils.widgets.get("job.catalog")
print(catalog)

# qqqq these var passings shouldnt be the best way need to see a working example
# So pytest can get it from the environment
os.environ["CATALOG"] = catalog
# This looks 'up' one level from this notebook to find your project root
# using toml now
# repo_root = os.path.abspath('..') # sys.path.append(f"{repo_root}/src")

# Prevent __pycache__ creation
sys.dont_write_bytecode = True

# print(f"Project root set to: {repo_root}")
print("Setup run")

## Unit Test Runner

In [0]:

#%skip
# Run your test file
# This will run every file starting with 'test_' in that folder
unit_exit_code = pytest.main([
    "unit-tests",
    "-v",                # verbose
    "-s",                # show print output
    "--tb=short"         # short tracebacks
])

print(f"Unit test exit code: {unit_exit_code}")

## Integration Test Runner

In [0]:
# Run your test file
# This will run every file starting with 'test_' in that folder
integration_exit_code = pytest.main([
    "integration-tests", 
    "-m", "not integration_sql", # exclude sql tests i want them to run seperately below
    "-v",                # verbose
    "-s",                # show print output
    "--tb=short"         # short tracebacks
])

print(f"Integration test exit code: {integration_exit_code}")

# Not sure we should do SQL based tests but this is an example
The pyton runs the sql test and then can fail the pipeline (which is what we want it to do as its acting as a testrunner)

Time may be better translating into python that unit testing it

In [0]:
# i think we should turn stored procedure into python functions, not test them this way but it depends on practicality
# there is an example of the function equivalent (close enough) in this solution

integration_sql_exit_code = pytest.main([
    "integration-tests", 
    "-m", "integration_sql", # exclude sql tests i want them to run seperately below
    "-v",                # verbose
    "-s",                # show print output
    "--tb=short"         # short tracebacks
])

print(f"Integration test exit code: {integration_sql_exit_code}")

# Fail or pass the pipeline


In [0]:
failures = {
    "unit": unit_exit_code,
    "integration": integration_exit_code,
    "integration-sql": integration_sql_exit_code
}

failed = {k: v for k, v in failures.items() if v != 0}

if failed:
    print("❌ Test failures detected:")
    for name, code in failed.items():
        print(f" - {name} tests failed with exit code {code}")

    # Fail the Databricks job
    raise RuntimeError(f"Tests failed: {failed}")

print("✅ All tests passed")

# Out of scope: code coverage
- Want to see direction code is going
- Like TELBlazor would be nice to inforce increase in coverage
  - and html report gitpage is a nice to have to